In [ ]:
# Updated import based on latest Langchain docs
from langchain_community.document_loaders import PyPDFLoader

# Task 3: The Loader Array
# Exact relative paths to your specific AI research papers
pdf_files = [
    "pdf_store_dir/html_cheatsheet.pdf",
    "pdf_store_dir/javascript_cheatsheet.pdf",
    "pdf_store_dir/mysql_cheatsheet.pdf",
]

# Task 4: The Aggregation Logic (Master Array)
master_array = []

# Loop through the files
for file_path in pdf_files:
    # Create the loader object
    loader = PyPDFLoader(file_path)
    
    # 🔥 THE COMBO TASK (Final Boss)
    # loader.load() returns a list of Document objects.
    # We use .extend() instead of .append() to keep the master_array strictly 1D (flat).
    # If we used .append(), it would create nested lists and crash the Vector DB later!
    master_array.extend(loader.load())

# --- Definition of Done (Verification) ---

# 1. Check the length (Target: 253)
print(f"Total pages loaded: {len(master_array)}")

# 2. Inspect the first item (Index 0) to verify page_content and metadata
if len(master_array) > 0:
    first_doc = master_array[0]
    print("\n--- First Item Inspection ---")
    print(f"Metadata: {first_doc.metadata}")
    print(f"Page Content Snippet: {first_doc.page_content[:200]}...\n")

In [ ]:
# Task 1: The Right Tool
# Updated import based on latest Langchain docs
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Task 2: Configuring the Dials
# We set our hard limit, our "recap" overlap, and turn on metadata tracking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True 
)

# 🔥 THE COMBO TASK (Final Boss)
# Pass the Langchain Document objects (master_array) into the splitter engine.
# We MUST use split_documents() to keep the page numbers and sources intact.
chunked_documents = text_splitter.split_documents(master_array)

# --- Definition of Done (Verification) ---

# 1. Check the length (Target is dynamic based on your cheat sheets)
print(f"\n--- 🔪 The Butcher Shop Results ---")
print(f"Total chunks created: {len(chunked_documents)}")

# 2. Inspect the first chunk to verify length limit and the new 'start_index'
if len(chunked_documents) > 0:
    first_chunk = chunked_documents[0]
    
    print("\n--- First Chunk Inspection ---")
    print(f"Chunk Character Length: {len(first_chunk.page_content)}")
    print(f"Metadata: {first_chunk.metadata}")
    print(f"Chunk Snippet: {first_chunk.page_content[:150]}...\n")

In [5]:
# Task 1: Model Initialization
# Updated import based on the latest Langchain architecture (Partner Package)
from langchain_ollama import OllamaEmbeddings

# Initialize the embedder with the specific Mistral model
embeddings = OllamaEmbeddings(model="mistral:7b")

# Task 2: The Vector Extraction
# We must pass strictly the string data (.page_content), NOT the whole Document object.
# If we passed chunked_documents[0] directly, we would get a fatal Type Error!
if len(chunked_documents) >= 2:
    text_1 = chunked_documents[0].page_content
    text_2 = chunked_documents[1].page_content
    
    print("\nTranslating chunks into the Matrix... (This might take a few seconds)")
    
    # Generate the vector embeddings
    vector_1 = embeddings.embed_query(text_1)
    vector_2 = embeddings.embed_query(text_2)

    # 🔥 THE COMBO TASK (Final Boss)
    # Strict pipeline validation: Python will throw an AssertionError and stop the script 
    # if the lengths do not match.
    assert len(vector_1) == len(vector_2), "FATAL ERROR: Vector dimensions do not match!"

    # --- Definition of Done (Verification) ---
    print("\n--- 🕶️ The Matrix Translation Results ---")
    print("✅ Assertion Passed: Both chunks have the exact same vector dimension.")
    # Mistral usually outputs a dimension size of 4096
    print(f"Mistral Vector Dimension Size: {len(vector_1)}") 
else:
    print("Error: Not enough chunks generated in the previous step to compare two vectors.")


Translating chunks into the Matrix... (This might take a few seconds)

--- 🕶️ The Matrix Translation Results ---
✅ Assertion Passed: Both chunks have the exact same vector dimension.
Mistral Vector Dimension Size: 4096


In [ ]:
# Task 1 & 2: The Vault Setup & Import
# Ekdum latest langchain-chroma class import kar rahe hain
from langchain_chroma import Chroma
import os

print("\n🏦 Vault Opening: Bhej rahe hain saare chunks ko Chroma DB mein...")
print("⏳ Warning: Isme thoda time lagega kyunki Mistral locally har chunk ko vector me badal raha hai (Go grab a coffee ☕).")

# 🔥 THE COMBO TASK (Final Boss)
# Factory method call kar rahe hain: Chroma.from_documents()
# Keyword strictly 'embedding' use kiya hai taaki schema error na aaye.

persist_dir = "./chroma_db"

vector_store = Chroma.from_documents(
    documents=chunked_documents,         # Tere chunks wala array (cheat sheets wala)
    embedding=embeddings,                # Mistral embedder jo humne Module 4 me banaya tha
    collection_name="cheat_sheets_vault", # Database me table ka naam
    persist_directory=persist_dir        # Hard drive pe folder jahan sqlite3 files save hongi
)

# --- Definition of Done (Verification) ---
print("\n--- 🏦 The Vault Verification ---")
if os.path.exists(persist_dir):
    print(f"✅ Success! Chroma DB folder '{persist_dir}' tere project me create ho gaya hai.")
    print("🎉 Saara data ab permanently teri hard drive par lock ho chuka hai!")
    print(f"Total chunks saved in Vault: {len(chunked_documents)}")
else:
    print("❌ Error: Folder create nahi hua. Kuch gadbad hai.")


🏦 Vault Opening: Bhej rahe hain saare chunks ko Chroma DB mein...
⏳ Warning: Isme thoda time lagega kyunki Mistral locally har chunk ko vector me badal raha hai (Go grab a coffee ☕).
